# Data Ingestions Types 

In [0]:

%sql
USE CATALOG workspace;
USE SCHEMA salesdb;
SELECT current_catalog() , current_schema()  

In [0]:
%sql

DROP TABLE IF EXISTS covid_bing; 

CREATE TABLE covid_bing
SELECT  * 
FROM read_files(
  '/Volumes/workspace/salesdb/covid/bing_covid-19_data.parquet',
  format => 'parquet'
) ;

SELECT * FROM covid_bing
LIMIT 10

In [0]:
df = (spark
      .read
      .format("parquet")
      .load('/Volumes/workspace/salesdb/covid/bing_covid-19_data.parquet')
    )

(df
  .write
  .mode("overwrite")
  .saveAsTable("workspace.salesdb.covid_bing")  
)

df = spark.table("workspace.salesdb.covid_bing")
display(df)

## # Incremental Ingestion With COPYINTO

In [0]:
%sql
    
DROP TABLE IF EXISTS covid_bing;
DROP TABLE IF EXISTS covid_bings_incre;

CREATE TABLE covid_bings_incre (dummy STRING);

copy into covid_bings_incre
from '/Volumes/workspace/salesdb/covid/bing_covid-19_data.parquet'
fileformat = parquet
copy_options ('mergeSchema' = 'true')

In [0]:
%sql
copy into covid_bings_incre
from '/Volumes/workspace/salesdb/covid/bing_covid-19_data.parquet'
fileformat = parquet
copy_options ('mergeSchema' = 'true')

# SQL AutoLoader

In [0]:
from pyspark.sql.functions import  col, current_timestamp

file_path = "/Volumes/workspace/salesdb/covid/"
table_name = "workspace.salesdb.covid_final"
checkpoint_path = "/Volumes/workspace/salesdb/covid/_checkpoint"

(spark
  .readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "parquet")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .option("header", "true")
  .option("inferSchema", "true")
  .load(file_path)
  .withColumn("_rescued_data", col("_rescued_data").cast("string"))
  .withColumn("load_time", current_timestamp())
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .table(table_name)
)


